# Lab 04 — Engenharia de características e regressão polinomial
**Curso 1 · Semana 2 · Machine Learning — Andrew Ng**

Adaptação didática em português de C1_W2_Lab04_FeatEng_PolyReg_Soln.ipynb. O original importa funções auxiliares do curso que não foram anexadas. Neste caderno implementamos uma descida do gradiente equivalente e a padronização, preservando as entradas, alvos e taxas das demonstrações. Os pesos impressos podem variar em relação ao original.

## 1. Objetivo

Até agora, uma regressão linear simples produz uma reta: $f_{w,b}(x)=wx+b$. Uma regressão múltipla produz uma combinação linear das características: $f_{\vec w,b}(\vec x)=\vec w\cdot\vec x+b$.

E se o comportamento real dos dados for uma curva? Em vez de abandonar a regressão linear, podemos transformar ou combinar as entradas. Isso é **feature engineering**: criar características que representem melhor o problema.

Exemplo conceitual: a partir de largura e comprimento de uma casa, criar a área = largura × comprimento. Na regressão polinomial, criamos potências como $x^2$ e $x^3$. A regressão continua linear **nos parâmetros** $w$ e $b$, mesmo que a previsão seja não linear em relação ao $x$ original.

Instale NumPy e Matplotlib antes de executar, se necessário: python -m pip install numpy matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## 2. Revisão da matemática e implementação das funções auxiliares

- $m$: número de exemplos; $n$: número de características.
- $X$: matriz de dimensão $(m,n)$; cada linha é um exemplo.
- $y$: alvo real, um vetor de tamanho $m$.
- $\hat y^{(i)}=\vec w\cdot\vec x^{(i)}+b$: previsão.
- $i$ é o índice do exemplo, não uma potência. Já $x^2$ é uma potência.
- $j$ identifica uma característica e o seu peso.

$$J(\vec w,b)=\frac{1}{2m}\sum_{i=0}^{m-1}(\hat y^{(i)}-y^{(i)})^2$$
$$\frac{\partial J}{\partial w_j}=\frac{1}{m}\sum_i(\hat y^{(i)}-y^{(i)})x_j^{(i)},\quad \frac{\partial J}{\partial b}=\frac{1}{m}\sum_i(\hat y^{(i)}-y^{(i)})$$

A descida do gradiente substitui as derivadas nas atualizações:
$$w_j:=w_j-\alpha\frac{1}{m}\sum_i(\hat y^{(i)}-y^{(i)})x_j^{(i)}$$
$$b:=b-\alpha\frac{1}{m}\sum_i(\hat y^{(i)}-y^{(i)})$$

O custo agrega erros quadráticos, enquanto o gradiente indica como mudar os parâmetros. Calculamos os dois gradientes com os parâmetros antigos, antes de atualizar qualquer um.

Na forma vetorizada, X @ w produz m previsões, X.T @ erros produz n somas de erro vezes característica. X.T troca as linhas pelas colunas, e dividir por m calcula a média das contribuições. O gradiente de b é a média simples dos erros. O histórico guarda valores de custo para comparar convergência.

In [ ]:
def calcular_custo(X, y, w, b):
    erros = X @ w + b - y
    return np.mean(erros**2) / 2

def calcular_gradiente(X, y, w, b):
    erros = X @ w + b - y
    return X.T @ erros / len(y), np.mean(erros)

def treinar(X, y, iteracoes, alpha):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    w = np.zeros(X.shape[1])
    b = 0.0
    historico = [calcular_custo(X, y, w, b)]
    for i in range(iteracoes):
        dw, db = calcular_gradiente(X, y, w, b)
        w = w - alpha * dw
        b = b - alpha * db
        if (i+1) % max(1, iteracoes//100) == 0 or i+1 == iteracoes:
            custo = calcular_custo(X, y, w, b)
            if not np.isfinite(custo):
                raise ValueError("Custo não finito: revise a taxa de aprendizado.")
            historico.append(custo)
    return w, b, np.array(historico)

def mostrar_ajuste(x, y, X, w, b, titulo):
    plt.scatter(x, y, marker="x", label="Real")
    plt.plot(x, X @ w + b, label="Previsto")
    plt.xlabel("x original")
    plt.ylabel("y")
    plt.title(titulo)
    plt.legend()
    plt.show()

def padronizar(X):
    media = X.mean(axis=0)
    desvio = X.std(axis=0)
    desvio = np.where(desvio == 0, 1, desvio)
    return (X-media)/desvio, media, desvio

## 3. Primeiro teste: tentar ajustar uma reta a uma parábola

O original cria 20 exemplos: x de 0 a 19, e $y=1+x^2$. np.arange(0,20,1) começa em zero, avança de um em um e exclui 20. O operador ** eleva cada elemento ao quadrado. reshape(-1,1) transforma o vetor (20,) em uma matriz (20,1): uma coluna, com a quantidade de linhas inferida por -1.

O modelo recebe apenas x e tenta ajustar $wx+b$. Mesmo com um treinamento correto, não pode reproduzir exatamente a curva. Uma linha tem inclinação constante; a parábola tem inclinação variável.

O laboratório usa 1000 iterações e alpha=0.01. O scatter mostra os alvos; plot desenha a previsão para cada entrada na ordem crescente de x. A discrepância é limitação da representação, e não necessariamente um erro da implementação.

In [ ]:
x = np.arange(0, 20, 1)
y = 1 + x**2
X = x.reshape(-1, 1)
w, b, hist = treinar(X, y, 1000, 1e-2)
print("w:", w, "b:", b, "custo final:", hist[-1])
mostrar_ajuste(x, y, X, w, b, "Sem engenharia de características")

## 4. Criar a característica quadrática

Agora o modelo recebe $z=x^2$ como entrada. A fórmula é $f=wz+b=wx^2+b$.

O alvo continua sendo $1+x^2$. Para x=3, a característica passa de 3 para 9, e o alvo continua 10. A solução exata para estes dados é w=1 e b=1. O treinamento busca se aproximar dela.

No original são 10.000 iterações com alpha=0.00001. Por que a taxa diminuiu? A escala da entrada aumentou: antes o máximo era 19, agora 361. Gradientes podem crescer muito e uma taxa anterior pode causar divergência.

O material registra aproximadamente w=1 e b=0.0490, um bom ajuste visual, mas não a solução exata. Um intercepto distante de 1 indica que ainda há espaço para melhorar o ajuste.

In [ ]:
X = (x**2).reshape(-1, 1)
w, b, hist = treinar(X, y, 10000, 1e-5)
print("w:", w, "b:", b, "custo:", hist[-1])
mostrar_ajuste(x, y, X, w, b, "Característica x²")

## 5. E se não soubermos qual potência usar?

O laboratório muda o alvo para $y=x^2$ (sem o +1). Criamos três colunas: $x$, $x^2$ e $x^3$. A função np.c_[x, x**2, x**3] junta vetores como colunas; para x=3 a linha é [3,9,27].

$$f(x)=w_0x+w_1x^2+w_2x^3+b$$

Note que há uma variável original e três características criadas. Os pesos são independentes. A solução exata para estes dados sem ruído é [0,1,0] e b=0. A taxa original é 1e-7 e são 10.000 passos.

O original registra aproximadamente [0.08,0.54,0.03] e b=0.0106. Esses valores mostram um ajuste intermediário; o coeficiente do termo quadrático ganhou destaque. **Cuidado:** não compare diretamente magnitudes de pesos com entradas em escalas diferentes, nem conclua causalidade ou seleção definitiva. Um peso pequeno multiplicado por um x³ muito grande ainda pode contribuir muito. Características correlacionadas também dificultam interpretar coeficientes.

In [ ]:
y = x**2
X = np.c_[x, x**2, x**3]
print("Formato:", X.shape)
print("Primeiras linhas:\n", X[:5])
w, b, hist = treinar(X, y, 10000, 1e-7)
print("w:", w, "b:", b, "custo:", hist[-1])
mostrar_ajuste(x, y, X, w, b, "Características x, x² e x³")

## 6. Visualização de cada característica versus o alvo

plt.subplots cria uma figura com três eixos. sharey=True compartilha a escala vertical, tornando a comparação mais clara. X[:,j] seleciona todas as linhas da coluna j. len(ax) conta quantos gráficos existem. A dispersão da coluna x² contra y forma uma reta porque são iguais neste exemplo. As outras colunas apresentam relações não lineares com y. Isso ajuda a pensar em transformações que tornam a relação com o alvo mais próxima de linear.

In [ ]:
nomes = ["x", "x²", "x³"]
fig, ax = plt.subplots(1, 3, figsize=(12,3), sharey=True)
for j in range(len(ax)):
    ax[j].scatter(X[:,j], y)
    ax[j].set_xlabel(nomes[j])
ax[0].set_ylabel("y")
plt.tight_layout()
plt.show()

## 7. Escalonamento das características polinomiais

x, x² e x³ possuem amplitudes 19, 361 e 6859. np.ptp(X,axis=0) calcula máximo menos mínimo de cada coluna. Essa diferença de escala é justamente o problema da aula anterior.

$$z_j^{(i)}=\frac{x_j^{(i)}-\mu_j}{\sigma_j}$$

A média e o desvio são calculados por coluna (axis=0). A subtração centraliza os dados; a divisão deixa cada coluna variável com desvio padrão 1. np.where impede divisão por zero nas colunas constantes. Não alteramos y.

Agora o original usa alpha=0.1 e 100.000 iterações. A taxa maior é possível neste exemplo graças à escala mais equilibrada. A convergência não é instantânea: as potências ainda são fortemente correlacionadas, e essa correlação dificulta a otimização.

Os pesos obtidos com dados padronizados estão em outra escala: não são comparáveis diretamente aos pesos sem normalização.

In [ ]:
X_norm, media, desvio = padronizar(X)
print("Amplitude original:", np.ptp(X, axis=0))
print("Amplitude normalizada:", np.ptp(X_norm, axis=0))
print("Média normalizada:", X_norm.mean(axis=0))
print("Desvio normalizado:", X_norm.std(axis=0))
w, b, hist = treinar(X_norm, y, 100000, 0.1)
print("w normalizado:", w, "b:", b, "custo:", hist[-1])
mostrar_ajuste(x, y, X_norm, w, b, "Regressão polinomial normalizada")
plt.plot(np.linspace(0, 100000, len(hist)), hist)
plt.yscale("log")
plt.xlabel("Iterações")
plt.ylabel("Custo J")
plt.show()

### Interpretando os coeficientes e fazendo uma nova previsão

Se o modelo recebe $z_j=(x_j-\mu_j)/\sigma_j$, então o peso equivalente na escala original é $w_j^{original}=w_j^{norm}/\sigma_j$ e o intercepto é $b^{original}=b^{norm}-\sum_j w_j^{original}\mu_j$. Isso permite verificar a contribuição das potências no modelo original. Para previsão de x=4, precisamos gerar as mesmas três características e usar exatamente as estatísticas do treino; não recalculamos média e desvio a partir do exemplo novo.

In [ ]:
w_original = w / desvio
b_original = b - w_original @ media
print("Pesos originais:", w_original, "b original:", b_original)
novo = np.array([4, 4**2, 4**3])
novo_norm = (novo - media)/desvio
print("Previsão:", novo_norm @ w + b)
print("Alvo esperado:", 4**2)
print("Previsão equivalente:", novo @ w_original + b_original)

## 8. Funções complexas: cosseno com 13 características

O último experimento usa $y=\cos(x/2)$ e potências de x do grau 1 ao 13. np.cos calcula o cosseno em radianos. np.column_stack reúne as 13 colunas e é equivalente à expressão longa np.c_[x,x**2,...,x**13] do original.

Usamos float antes de gerar as potências para evitar overflow de inteiros em outras faixas. A regressão aprende uma combinação polinomial, não o cosseno em si. O professor usa 1.000.000 de iterações e alpha=0.1; o código abaixo começa com 100.000 para uma execução mais curta, e você pode mudar a variável para 1.000.000. O ajuste pode melhorar, mas polinômios de grau alto podem oscilar e extrapolar mal. O gráfico só mede o ajuste nos pontos de treino.

In [ ]:
x = np.arange(0,20,dtype=float)
y = np.cos(x/2)
X = np.column_stack([x**grau for grau in range(1,14)])
X_norm, media, desvio = padronizar(X)
iteracoes = 100_000  # original: 1_000_000
w, b, hist = treinar(X_norm, y, iteracoes, 0.1)
print("Formato:", X_norm.shape, "custo final:", hist[-1])
mostrar_ajuste(x, y, X_norm, w, b, "Aproximação polinomial de cos(x/2)")

## 9. Revisão e exercícios

- Engenharia de características transforma as entradas, não os alvos.
- Regressão polinomial é linear nos parâmetros e não linear no x original.
- Escalonamento facilita o gradiente, principalmente com potências.
- Custo baixo no treino não garante bom desempenho em exemplos novos. Selecione grau e regularização usando validação; não use o teste para escolher o modelo.

**Exercícios:** (1) Para x=5, quais as três características? (2) Por que o modelo wx+b não pode ajustar exatamente y=x² em 20 pontos? (3) Qual a diferença entre x^(i) e x²? (4) Por que não padronizar uma nova casa isoladamente? (5) Mude o grau do cosseno para 3 e compare.

**Respostas:** (1) 5,25,125. (2) Uma reta tem inclinação constante. (3) O índice entre parênteses indica o exemplo; o expoente 2 é uma potência. (4) Os parâmetros aprendidos dependem das estatísticas do treino. (5) Execute e compare custos e gráficos; grau maior não implica melhor generalização.

Fonte: laboratório opcional de Feature Engineering and Polynomial Regression do curso de Andrew Ng (DeepLearning.AI / Stanford Online), arquivo enviado pelo aluno.